# # Ep 03 — Structured Output with Pydantic

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model


In [2]:
load_dotenv()

True

In [3]:
OPENAI_MODEL = os.getenv("OPENAI_MODEl", "openai:gpt-5.6-luna")
GROQ_MODEL = os.getenv("GROPQ_MODEL", "GROQ:meta-llama/llama-prompt-guard-2-22m")

print("Openai model: ", OPENAI_MODEL)
print("Groq model: ", GROQ_MODEL)

Openai model:  openai:gpt-5-nano
Groq model:  groq:qwen/qwen3.8-27b


## Ouput structure format using Pydantic

In [5]:
class Resume(BaseModel):
    name: str = Field(description="Full name of the candidate")
    email: str | None = Field(description="Get email if present, else none")
    years_experience: float = Field(description="Total years of professional experience")
    skills: list[str] = Field(description="List of techinical skills mentioned")

In [8]:
Resume(name="Raj", email="raj@gmail.com", years_experience=5.9, skills=["Python", "Langchain", "Langgraph"])

Resume(name='Raj', email='raj@gmail.com', years_experience=5.9, skills=['Python', 'Langchain', 'Langgraph'])

In [11]:
#Resume(name=["Raj"], email="raj@gmail.com", years_experience=5, skills=["Python", "Langchain", "Langgraph"])

## Load input data

In [18]:
SAMPLE_DIR = Path("samples")

text = (SAMPLE_DIR / "resume_messy.txt").read_text(encoding="utf-8")

In [20]:
print(text)

hey so my name is Aarav Sharma, been working as a backend dev for about 4 and a half
years now. you can reach me at aarav.sharma@example.com or just ping on linkedin.
mostly i do python, fastapi, postgres, some docker and aws. recently started learning
langgraph and building ai agents which is super fun. also know a bit of react but not my
main thing. looking for agentic ai engineer roles now.



## Parse resume using LLM

In [22]:
llm = init_chat_model(OPENAI_MODEL, temperature=0) ## 

In [23]:
# question: "Do you love Python?"

# temperature : 0 >>> "I love python"

# temperature : 0.5 >>> "I love python too much"

# temperature : 0.9 >>> "I love programming language pyhon because i learned it"

# temperature : 1 >>> "I love pyhon because i love animals"

In [ ]:
prompt = f"From given resume {text}  /n extract name, email, years of experience and skils in json format."
responce = llm.invoke(prompt)

In [28]:
responce

AIMessage(content='{\n  "name": "Aarav Sharma",\n  "email": "aarav.sharma@example.com",\n  "years_of_experience": 4.5,\n  "skills": [\n    "Python",\n    "FastAPI",\n    "PostgreSQL",\n    "Docker",\n    "AWS",\n    "LangGraph",\n    "AI agents",\n    "React (basic)"\n  ]\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 859, 'prompt_tokens': 122, 'total_tokens': 981, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ERGvygWyIXbjpXTvIoaCfyZ1c37rq', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0ce5d-efc2-7491-8faa-ade2004a1608-0', tool_calls=[],

In [29]:
responce.content

'{\n  "name": "Aarav Sharma",\n  "email": "aarav.sharma@example.com",\n  "years_of_experience": 4.5,\n  "skills": [\n    "Python",\n    "FastAPI",\n    "PostgreSQL",\n    "Docker",\n    "AWS",\n    "LangGraph",\n    "AI agents",\n    "React (basic)"\n  ]\n}'

In [30]:
llm_structured = llm.with_structured_output(Resume, include_raw=True)

In [ ]:
# llm_structured.invoke(prompt)

## Messages: system vs user

System message = the agent's role/rules ("You are a precise data extractor.").
User message = the actual request/content. Good system prompts = more reliable output. (We'll harden these as guardrails in Ep 35.)

In [31]:
system = "You are a precise data extractor. Extract only what is present; never invent data."

messges = [
{"role": "system", "content": system},
{"role": "user", "content": f"Extract structured data from this resume : \n {text}"}
]

responce = llm_structured.invoke(messges)


In [32]:
responce

{'raw': AIMessage(content='{"name":"Aarav Sharma","email":"aarav.sharma@example.com","years_experience":4.5,"skills":["python","fastapi","postgres","docker","aws","react","langgraph","ai agents"]}', additional_kwargs={'parsed': Resume(name='Aarav Sharma', email='aarav.sharma@example.com', years_experience=4.5, skills=['python', 'fastapi', 'postgres', 'docker', 'aws', 'react', 'langgraph', 'ai agents']), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2683, 'prompt_tokens': 250, 'total_tokens': 2933, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2624, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ERH6RkZCyqTGeBjQ5oh7wLAscgjos', 'service_tier': 'default'

In [33]:
type(responce)

dict

In [35]:
responce["parsed"]

Resume(name='Aarav Sharma', email='aarav.sharma@example.com', years_experience=4.5, skills=['python', 'fastapi', 'postgres', 'docker', 'aws', 'react', 'langgraph', 'ai agents'])

### TOEKN/CONT awareness

In [ ]:
{'raw': AIMessage(content='{"name":"Aarav Sharma","email":"aarav.sharma@example.com",
"years_experience":4.5,"skills":["python","fastapi","postgres","docker","aws","react",
"langgraph","ai agents"]}', additional_kwargs={'parsed': Resume(name='Aarav Sharma', 
email='aarav.sharma@example.com', years_experience=4.5, skills=['python', 'fastapi', 
'postgres', 'docker', 'aws', 'react', 'langgraph', 'ai agents']), 'refusal': None},
 response_metadata={'token_usage': {'completion_tokens': 2683, 'prompt_tokens': 250, 
 'total_tokens': 2933, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 
 'audio_tokens': 0, 'reasoning_tokens': 2624, 'rejected_prediction_tokens': 0, 
 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 
 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 
 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 
 'chatcmpl-ERH6RkZCyqTGeBjQ5oh7wLAscgjos', 'service_tier': 'default', 'finish_reason': 
 'stop', 'logprobs': None}, id='lc_run--01a0ce67-d240-7983-a8bf-ca0266aca9a5-0', 
 tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 250, 
 'output_tokens': 2683, 'total_tokens': 2933, 'input_token_details': 
 {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 2624}}),


In [37]:
usage = getattr(responce.get("raw"), "usage_metadata", None)
if usage:
    print(f"tokens — in: {usage.get('input_tokens')} out: {usage.get('output_tokens')} "
          f"total: {usage.get('total_tokens')}")

tokens — in: 250 out: 2683 total: 2933


## never trust unvalidated ouput

In [39]:
if responce.get("parsing_error"):
    print("Model output failed validation; handle/retry instead of trusting it.")
else:
    resume = responce.get("parsed")
    print(resume.model_dump_json(indent=2))
    print(f"\nValidated! {len(resume.skills)} skills, {resume.years_experience} yrs experience.")

{
  "name": "Aarav Sharma",
  "email": "aarav.sharma@example.com",
  "years_experience": 4.5,
  "skills": [
    "python",
    "fastapi",
    "postgres",
    "docker",
    "aws",
    "react",
    "langgraph",
    "ai agents"
  ]
}

Validated! 8 skills, 4.5 yrs experience.
